<a href="https://colab.research.google.com/github/jaysulk/GENERIC-FNO/blob/main/GENERIC_FNO_4_PDE_Long_Horizon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#!/usr/bin/env python3
# ============================================================================
# GENERIC-FNO LONG-HORIZON STABILITY -- ONE SELF-CONTAINED COLAB CELL.
# Nothing else to run. Paste this whole thing in a single cell and execute.
# Part A defines the model classes + PDE generators; Part C trains FNO / EP-FNO
# / GENERIC on heat, advection, Burgers and rolls each out far past the 15-step
# training horizon, saving fig_long_horizon.{pdf,png}.
# Optional: set os.environ["GENERIC_FNO_FIGDIR"]=... and LH_RES / LH_EPOCHS etc.
# BEFORE this cell. To reuse models you already trained, call
#   run_long_horizon(pretrained={(pde, kind): model, ...})  instead of the
# auto-run at the bottom.
# ============================================================================

# ==================== PART A: MODELS + GENERATORS (verbatim) =================
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pickle
import time
import math
from collections import defaultdict


# ============================================================================
# Data Generation — 2D PDEs (spectral)
# ============================================================================

def _random_field_2d(nx, ny, max_mode, n_modes, amp_scale=0.5, device='cpu'):
    """Build a random smooth 2D field as a sum of sine modes."""
    x = torch.linspace(0, 2*math.pi, nx+1, device=device)[:-1]
    y = torch.linspace(0, 2*math.pi, ny+1, device=device)[:-1]
    X, Y = torch.meshgrid(x, y, indexing='ij')
    u = torch.zeros(nx, ny, device=device)
    for _ in range(n_modes):
        kx = torch.randint(1, max_mode, (1,)).item()
        ky = torch.randint(1, max_mode, (1,)).item()
        amp = torch.randn(1).item() * amp_scale
        phase = torch.rand(1).item() * 2 * math.pi
        u += amp * torch.sin(kx * X + ky * Y + phase)
    return u


def generate_heat_data_2d(n_samples=150, nx=128, nt=15, dt=0.005, nu=0.02, device='cpu'):
    """2D heat: du/dt = nu*(uxx+uyy). Purely dissipative. Exact in spectral space."""
    kx = torch.fft.fftfreq(nx, d=1.0/nx).to(device)
    ky = torch.fft.rfftfreq(nx, d=1.0/nx).to(device)
    KX, KY = torch.meshgrid(kx, ky, indexing='ij')
    k_sq = KX**2 + KY**2
    decay = torch.exp(-nu * k_sq * dt)

    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(3, 7, (1,)).item()
        u0 = _random_field_2d(nx, nx, nx//8, n_modes, device=device)
        u_hat = torch.fft.rfft2(u0)
        traj = [u0.clone()]
        for t in range(nt):
            u_hat = u_hat * decay
            traj.append(torch.fft.irfft2(u_hat, s=(nx, nx)))
        traj = torch.stack(traj, dim=0)  # (nt+1, nx, nx)
        data_in.append(traj[:-1])
        data_out.append(traj[1:])
    return torch.stack(data_in), torch.stack(data_out), 'heat'


def generate_wave_data_2d(n_samples=150, nx=128, nt=15, dt=0.005, c=1.0, device='cpu'):
    """2D wave: u_tt = c²(uxx+uyy). Reversible. Track u-component, exact spectral rotation."""
    kx = torch.fft.fftfreq(nx, d=1.0/nx).to(device)
    ky = torch.fft.rfftfreq(nx, d=1.0/nx).to(device)
    KX, KY = torch.meshgrid(kx, ky, indexing='ij')
    kmag = torch.sqrt(KX**2 + KY**2)
    omega = c * kmag

    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(3, 7, (1,)).item()
        u0 = _random_field_2d(nx, nx, nx//8, n_modes, device=device)
        v0 = _random_field_2d(nx, nx, nx//8, n_modes, amp_scale=0.3, device=device)
        u_hat = torch.fft.rfft2(u0)
        v_hat = torch.fft.rfft2(v0)
        traj = [u0.clone()]
        for t in range(nt):
            cos_w = torch.cos(omega * dt)
            sin_w = torch.sin(omega * dt)
            # rotate (u, v) preserving energy; guard omega=0 mode
            safe_omega = torch.where(omega > 1e-8, omega, torch.ones_like(omega))
            u_new = cos_w * u_hat + (sin_w / safe_omega) * v_hat
            v_new = -safe_omega * sin_w * u_hat + cos_w * v_hat
            u_hat, v_hat = u_new, v_new
            traj.append(torch.fft.irfft2(u_hat, s=(nx, nx)))
        traj = torch.stack(traj, dim=0)
        data_in.append(traj[:-1])
        data_out.append(traj[1:])
    return torch.stack(data_in), torch.stack(data_out), 'wave'


def generate_advection_data_2d(n_samples=150, nx=128, nt=15, dt=0.005, c=1.0,
                               max_mode=6, device='cpu'):
    """2D linear advection: u_t + c(u_x+u_y) = 0. Reversible, Markovian in u,
    conserves 0.5<u^2> exactly. Clean fully-observed reversible scalar test
    => a thermodynamically-consistent operator should drive M -> 0.
    Band-limited to max_mode (fixed, NOT nx//8) so the content stays within the
    operator's mode range and the per-step phase rotation is small -- otherwise
    high-k transport aliases and even a plain FNO fails (the operators only span
    the lowest modes_op modes)."""
    kx = torch.fft.fftfreq(nx, d=1.0/nx).to(device)
    ky = torch.fft.rfftfreq(nx, d=1.0/nx).to(device)
    KX, KY = torch.meshgrid(kx, ky, indexing='ij')
    phase = torch.exp(-1j * c * (KX + KY) * dt)
    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(3, 7, (1,)).item()
        u0 = _random_field_2d(nx, nx, max_mode, n_modes, device=device)
        u_hat = torch.fft.rfft2(u0)
        traj = [u0.clone()]
        for t in range(nt):
            u_hat = u_hat * phase
            traj.append(torch.fft.irfft2(u_hat, s=(nx, nx)))
        traj = torch.stack(traj, dim=0)
        data_in.append(traj[:-1])
        data_out.append(traj[1:])
    return torch.stack(data_in), torch.stack(data_out), 'advection'


def generate_burgers_data_2d(n_samples=150, nx=128, nt=15, dt=0.002, nu=0.02, device='cpu'):
    """2D scalar Burgers: u_t + u(u_x+u_y) = nu*(uxx+uyy). Mixed rev+diss.
    Semi-implicit: diffusion in spectral, advection explicit."""
    kx = torch.fft.fftfreq(nx, d=1.0/nx).to(device)
    ky = torch.fft.rfftfreq(nx, d=1.0/nx).to(device)
    KX, KY = torch.meshgrid(kx, ky, indexing='ij')
    k_sq = KX**2 + KY**2

    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(2, 5, (1,)).item()
        u = _random_field_2d(nx, nx, 5, n_modes, amp_scale=0.3, device=device)
        traj = [u.clone()]
        for t in range(nt):
            u_hat = torch.fft.rfft2(u)
            # implicit diffusion
            u_hat = u_hat / (1 + nu * k_sq * dt)
            u = torch.fft.irfft2(u_hat, s=(nx, nx))
            # explicit advection (spectral derivatives)
            ux = torch.fft.irfft2(1j * KX * torch.fft.rfft2(u), s=(nx, nx))
            uy = torch.fft.irfft2(1j * KY * torch.fft.rfft2(u), s=(nx, nx))
            u = u - dt * u * (ux + uy)
            traj.append(u.clone())
        traj = torch.stack(traj, dim=0)
        data_in.append(traj[:-1])
        data_out.append(traj[1:])
    return torch.stack(data_in), torch.stack(data_out), 'burgers'


# ============================================================================
# Building Blocks — 2D
# ============================================================================

class SpectralConv2d(nn.Module):
    """Standard FNO spectral convolution (2D). Two corners for +/- kx."""
    def __init__(self, in_ch, out_ch, modes1, modes2):
        super().__init__()
        self.modes1 = modes1  # kx modes (keep both +/- corners)
        self.modes2 = modes2  # ky modes (non-negative only, rfft)
        scale = 1.0 / (in_ch * out_ch)
        self.W1 = nn.Parameter(scale * torch.randn(out_ch, in_ch, modes1, modes2, dtype=torch.cfloat))
        self.W2 = nn.Parameter(scale * torch.randn(out_ch, in_ch, modes1, modes2, dtype=torch.cfloat))

    def forward(self, x):
        B, C, H, W = x.shape
        x_hat = torch.fft.rfft2(x, dim=(-2, -1))  # (B, C, H, W//2+1)
        out_hat = torch.zeros(B, self.W1.shape[0], H, W // 2 + 1,
                              dtype=torch.cfloat, device=x.device)
        m1 = min(self.modes1, H // 2)
        m2 = min(self.modes2, W // 2 + 1)
        # top-left corner (positive kx)
        out_hat[:, :, :m1, :m2] = torch.einsum(
            'bixy,oixy->boxy', x_hat[:, :, :m1, :m2], self.W1[:, :, :m1, :m2])
        # bottom-left corner (negative kx)
        out_hat[:, :, -m1:, :m2] = torch.einsum(
            'bixy,oixy->boxy', x_hat[:, :, -m1:, :m2], self.W2[:, :, :m1, :m2])
        return torch.fft.irfft2(out_hat, s=(H, W))


class AAGELU(nn.Module):
    """Anti-aliased GELU. A pointwise nonlinearity injects high-frequency harmonics
    that ALIAS on a coarse grid, which is the main reason FNO-style nets are only
    approximately resolution-invariant. We upsample by `factor` (band-limited, via
    FFT zero-pad), apply GELU on the finer grid, then downsample (FFT truncate),
    which suppresses the aliased content. Operates on whatever (H,W) it receives,
    so it stays resolution-agnostic. Assumes even H,W (true for our grids)."""
    def __init__(self, factor=2):
        super().__init__()
        self.f = factor

    def forward(self, x):
        f = self.f
        if f == 1:
            return F.gelu(x)
        B, C, H, W = x.shape
        Xs = torch.fft.fftshift(torch.fft.fft2(x, dim=(-2, -1)), dim=(-2, -1))
        Hf, Wf = H * f, W * f
        ph, pw = (Hf - H) // 2, (Wf - W) // 2
        up = F.pad(Xs, (pw, Wf - W - pw, ph, Hf - H - ph))
        x_up = torch.fft.ifft2(torch.fft.ifftshift(up, dim=(-2, -1)), dim=(-2, -1)).real * (f * f)
        x_up = F.gelu(x_up)
        Ys = torch.fft.fftshift(torch.fft.fft2(x_up, dim=(-2, -1)), dim=(-2, -1))
        crop = Ys[..., ph:ph + H, pw:pw + W]
        return torch.fft.ifft2(torch.fft.ifftshift(crop, dim=(-2, -1)), dim=(-2, -1)).real / (f * f)


def _act(antialias):
    return AAGELU(2) if antialias else nn.GELU()


class FNO_Block2d(nn.Module):
    def __init__(self, width, modes1, modes2, antialias=False):
        super().__init__()
        self.conv = SpectralConv2d(width, width, modes1, modes2)
        self.skip = nn.Conv2d(width, width, 1)
        self.norm = nn.InstanceNorm2d(width)
        self.act = _act(antialias)

    def forward(self, x):
        return self.act(self.norm(self.conv(x) + self.skip(x)))


class FNO_Backbone2d(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, width=32, modes=16, n_layers=4, antialias=False):
        super().__init__()
        self.lift = nn.Conv2d(in_ch, width, 1)
        self.blocks = nn.ModuleList([FNO_Block2d(width, modes, modes, antialias=antialias)
                                     for _ in range(n_layers)])
        self.proj = nn.Sequential(
            nn.Conv2d(width, width, 1),
            _act(antialias),
            nn.Conv2d(width, out_ch, 1)
        )

    def forward(self, x):
        x = self.lift(x)
        for block in self.blocks:
            x = block(x)
        return self.proj(x)


class FunctionalNet2d(nn.Module):
    """FNO2d backbone → scalar functional F[u]. u:(B,1,H,W) → (B,)."""
    def __init__(self, width=24, modes=12, n_layers=3, antialias=False):
        super().__init__()
        self.backbone = FNO_Backbone2d(in_ch=1, out_ch=1, width=width,
                                        modes=modes, n_layers=n_layers, antialias=antialias)
        self.head = nn.Sequential(
            nn.Linear(1, 16),
            nn.GELU(),
            nn.Linear(16, 1)
        )

    def forward(self, u):
        density = self.backbone(u)              # (B,1,H,W)
        integral = density.mean(dim=(-1, -2))   # (B,1) — spatial average ∝ integral
        return self.head(integral).squeeze(-1)  # (B,)


# ============================================================================
# Model 1: Vanilla FNO (2D, residual)
# ============================================================================

class VanillaFNO2d(nn.Module):
    def __init__(self, width=32, modes=16, n_layers=4):
        super().__init__()
        self.backbone = FNO_Backbone2d(in_ch=1, out_ch=1, width=width,
                                        modes=modes, n_layers=n_layers)

    def forward(self, u):
        return u + self.backbone(u)

    def predict_with_info(self, u):
        return self.forward(u), {}


# ============================================================================
# Model 2: EP-FNO (2D, energy penalty)
# ============================================================================

class EP_FNO2d(nn.Module):
    def __init__(self, width=32, modes=16, n_layers=4):
        super().__init__()
        self.backbone = FNO_Backbone2d(in_ch=1, out_ch=1, width=width,
                                        modes=modes, n_layers=n_layers)

    def forward(self, u):
        return u + self.backbone(u)

    def predict_with_info(self, u):
        u_next = self.forward(u)
        E_in = 0.5 * (u**2).mean(dim=(-1, -2)).mean(dim=-1)
        E_out = 0.5 * (u_next**2).mean(dim=(-1, -2)).mean(dim=-1)
        return u_next, {'dE': E_out - E_in}

    def energy_penalty(self, info, pde_type):
        dE = info['dE']
        if pde_type in ('heat', 'burgers'):
            return (F.relu(dE)**2).mean()
        elif pde_type in ('wave', 'advection'):
            return (dE**2).mean()
        return torch.tensor(0.0, device=dE.device)


# ============================================================================
# Model 3: GENERIC-FNO (2D)
# ============================================================================

class GENERIC_FNO2d(nn.Module):
    """
    du/dt = L·δE/δu + M·δS/δu  with hard projection.
    L(kx,ky) = i·a  (anti-Hermitian diagonal), M(kx,ky) = |b|² (PSD diagonal).
    Operators are defined on the lowest (modes_op) modes in each direction,
    using two corners for +/- kx (rfft2 layout) → resolution invariant.
    """
    def __init__(self, nx=128, width_func=24, modes_func=12, n_layers_func=3,
                 modes_op=16, residual_gate_init=-3.0, l2_vargrad=False,
                 use_residual=True, degeneracy_construction=True,
                 antialias=False, integrator='euler'):
        super().__init__()
        self.nx = nx
        self.modes_op = modes_op
        # degeneracy_construction (DEFAULT, the thermodynamically-consistent model):
        #   build L = (I-P_S) D_L (I-P_S) and M = (I-P_E) D_M (I-P_E), where P_E,P_S
        #   are rank-1 projections onto delta E/delta u, delta S/delta u and D_L=i*a,
        #   D_M=|b|^2 are diagonal Fourier multipliers. Then L dS = 0 and M dE = 0
        #   EXACTLY, so energy is conserved (dE/dt=0) and entropy is produced
        #   (dS/dt=<dS,M dS> >= 0) by construction in ANY dimension -- no energy
        #   projection, no entropy correction, no free residual. A reversible PDE
        #   (wave) is forced to learn M->0 because dissipation can no longer hide
        #   behind a projection. Set False for the legacy projection+correction path
        #   (kept only for ablation; it does NOT specialize -- M and L are
        #   interchangeable under the projection, so everything routes through M).
        # l2_vargrad: use the L2 variational derivative (N/|Omega|) grad_u E. Affects
        #   only the operator-output SCALE (projections are scale-invariant ratios);
        #   harmless either way under the construction. Default off.
        # use_residual: only meaningful in the legacy path; a free residual would
        #   break the thermodynamic guarantee, so it is ignored when
        #   degeneracy_construction=True.
        self.degeneracy_construction = degeneracy_construction
        self.l2_vargrad = l2_vargrad
        self.use_residual = use_residual
        self.integrator = integrator   # 'euler' (default) or 'rk4' (norm-preserving)
        m1 = min(modes_op, nx // 2)
        m2 = min(modes_op, nx // 2 + 1)
        self.m1, self.m2 = m1, m2

        self.E_net = FunctionalNet2d(width=width_func, modes=modes_func,
                                     n_layers=n_layers_func, antialias=antialias)
        self.S_net = FunctionalNet2d(width=width_func, modes=modes_func,
                                     n_layers=n_layers_func, antialias=antialias)

        # L: anti-Hermitian diagonal, two kx corners
        self.a_pos = nn.Parameter(0.3 * torch.randn(m1, m2))
        self.a_neg = nn.Parameter(0.3 * torch.randn(m1, m2))
        # M: PSD diagonal, two kx corners (parameterized as |b|²)
        self.b_pos_r = nn.Parameter(0.3 * torch.randn(m1, m2))
        self.b_pos_i = nn.Parameter(0.3 * torch.randn(m1, m2))
        self.b_neg_r = nn.Parameter(0.3 * torch.randn(m1, m2))
        self.b_neg_i = nn.Parameter(0.3 * torch.randn(m1, m2))

        # Small gated residual for high-freq content
        self.residual = nn.Sequential(
            nn.Conv2d(1, 16, 1),
            nn.GELU(),
            nn.Conv2d(16, 1, 1)
        )
        self.residual_gate = nn.Parameter(torch.tensor(float(residual_gate_init)))

    def _apply_operators(self, dEdu_hat, dSdu_hat, H, W):
        """Apply diagonal L and M in 2D Fourier space (two kx corners)."""
        m1, m2 = self.m1, self.m2
        rev_hat = torch.zeros_like(dEdu_hat)
        diss_hat = torch.zeros_like(dSdu_hat)

        # L = i*a  (reversible)
        rev_hat[:, :, :m1, :m2] = 1j * self.a_pos * dEdu_hat[:, :, :m1, :m2]
        rev_hat[:, :, -m1:, :m2] = 1j * self.a_neg * dEdu_hat[:, :, -m1:, :m2]

        # M = |b|²  (dissipative, PSD)
        M_pos = self.b_pos_r**2 + self.b_pos_i**2
        M_neg = self.b_neg_r**2 + self.b_neg_i**2
        diss_hat[:, :, :m1, :m2] = M_pos * dSdu_hat[:, :, :m1, :m2]
        diss_hat[:, :, -m1:, :m2] = M_neg * dSdu_hat[:, :, -m1:, :m2]

        return rev_hat, diss_hat

    # --- single-operator Fourier multipliers (for degeneracy-by-construction) ---
    def _L_apply(self, v, H, W):
        """Apply the skew diagonal operator D_L = i*a to physical field v."""
        m1, m2 = self.m1, self.m2
        vh = torch.fft.rfft2(v, dim=(-2, -1))
        out = torch.zeros_like(vh)
        out[:, :, :m1, :m2] = 1j * self.a_pos * vh[:, :, :m1, :m2]
        out[:, :, -m1:, :m2] = 1j * self.a_neg * vh[:, :, -m1:, :m2]
        return torch.fft.irfft2(out, s=(H, W))

    def _M_apply(self, v, H, W):
        """Apply the PSD diagonal operator D_M = |b|^2 to physical field v."""
        m1, m2 = self.m1, self.m2
        vh = torch.fft.rfft2(v, dim=(-2, -1))
        out = torch.zeros_like(vh)
        Mp = self.b_pos_r**2 + self.b_pos_i**2
        Mn = self.b_neg_r**2 + self.b_neg_i**2
        out[:, :, :m1, :m2] = Mp * vh[:, :, :m1, :m2]
        out[:, :, -m1:, :m2] = Mn * vh[:, :, -m1:, :m2]
        return torch.fft.irfft2(out, s=(H, W))

    @staticmethod
    def _remove(v, w):
        """(I - P_w) v: remove the component of v along direction w, per sample.
        Scale-invariant in w (ratio), so the L2-vs-Euclidean choice is irrelevant."""
        ip = (v * w).sum(dim=(-1, -2), keepdim=True)
        nn = (w * w).sum(dim=(-1, -2), keepdim=True) + 1e-12
        return v - (ip / nn) * w

    def _generic_rhs(self, dEdu, dSdu, H, W):
        """du/dt = (I-P_S) D_L (I-P_S) dE + (I-P_E) D_M (I-P_E) dS.
        Degeneracy (L dS = 0, M dE = 0) holds exactly => dE/dt = 0 and
        dS/dt = <dS, M dS> >= 0 by construction, no projection needed."""
        rev = self._remove(self._L_apply(self._remove(dEdu, dSdu), H, W), dSdu)
        diss = self._remove(self._M_apply(self._remove(dSdu, dEdu), H, W), dEdu)
        return rev, diss

    def _field(self, u, H, W):
        """Reversible+dissipative increment rev+diss at state u (degeneracy path).
        u must require grad (RK4 intermediate states do). create_graph follows
        training so gradients flow through the integrator stages when training and
        eval stays memory-light."""
        cg = self.training
        E = self.E_net(u)
        S = self.S_net(u)
        dEdu = torch.autograd.grad(E.sum(), u, create_graph=cg)[0]
        dSdu = torch.autograd.grad(S.sum(), u, create_graph=cg)[0]
        if self.l2_vargrad:
            scale = (H * W) / (2.0 * math.pi) ** 2
            dEdu = dEdu * scale
            dSdu = dSdu * scale
        rev, diss = self._generic_rhs(dEdu, dSdu, H, W)
        return rev + diss

    def _rk4_step(self, u, H, W):
        """4th-order Runge-Kutta on the learned increment field. RK4's stability
        region contains a segment of the imaginary axis, so it suppresses the
        explicit-Euler amplitude growth of the skew (reversible) operator and
        tightens the finite-step energy drift to O(dt^5). Degeneracy
        (<dE,f>=0, <dS,Mf>>=0) holds at every stage, so the structural guarantees
        are unchanged; only the integration of them improves."""
        u0 = u.detach().requires_grad_(True)
        k1 = self._field(u0, H, W)
        k2 = self._field(u0 + 0.5 * k1, H, W)
        k3 = self._field(u0 + 0.5 * k2, H, W)
        k4 = self._field(u0 + k3, H, W)
        dudt = (k1 + 2.0 * k2 + 2.0 * k3 + k4) / 6.0
        return u + dudt

    def forward(self, u):
        B, C, H, W = u.shape

        if self.degeneracy_construction and self.integrator == 'rk4':
            return self._rk4_step(u, H, W)

        u_leaf = u.detach().requires_grad_(True)

        E = self.E_net(u_leaf)
        S = self.S_net(u_leaf)
        dEdu = torch.autograd.grad(E.sum(), u_leaf, create_graph=True)[0]
        dSdu = torch.autograd.grad(S.sum(), u_leaf, create_graph=True)[0]

        if self.l2_vargrad:
            # L2 variational derivative: delta E/delta u = (N/|Omega|) grad_u E.
            scale = (H * W) / (2.0 * math.pi) ** 2
            dEdu = dEdu * scale
            dSdu = dSdu * scale

        if self.degeneracy_construction:
            # Thermodynamically-consistent path: degeneracy by construction (Euler).
            rev, diss = self._generic_rhs(dEdu, dSdu, H, W)
            dudt = rev + diss
            return u + dudt

        # --- legacy projection + correction path (ablation only) ---
        dEdu_hat = torch.fft.rfft2(dEdu, dim=(-2, -1))
        dSdu_hat = torch.fft.rfft2(dSdu, dim=(-2, -1))

        rev_hat, diss_hat = self._apply_operators(dEdu_hat, dSdu_hat, H, W)
        rev = torch.fft.irfft2(rev_hat, s=(H, W))
        diss = torch.fft.irfft2(diss_hat, s=(H, W))

        dudt = rev + diss
        dudt = self._project_energy_conservation(dudt, dEdu)
        dudt = self._ensure_entropy_production(dudt, dSdu, dEdu)

        if self.use_residual:
            gate = torch.sigmoid(self.residual_gate)
            residual = gate * self.residual(u_leaf)
            residual = self._project_energy_conservation(residual, dEdu)
            dudt = dudt + residual

        return u + dudt

    def _project_energy_conservation(self, dudt, dEdu):
        """Project du/dt ⊥ δE/δu over both spatial dims → dE/dt = 0."""
        inner = (dudt * dEdu).sum(dim=(-1, -2), keepdim=True)
        norm_sq = (dEdu * dEdu).sum(dim=(-1, -2), keepdim=True) + 1e-10
        return dudt - (inner / norm_sq) * dEdu

    def _ensure_entropy_production(self, dudt, dSdu, dEdu):
        """Ensure dS/dt = ⟨δS/δu, du/dt⟩ ≥ 0 via correction ⊥ δE/δu."""
        dSdt = (dSdu * dudt).sum(dim=(-1, -2), keepdim=True)
        violation = F.relu(-dSdt)
        if violation.sum() > 0:
            inner_SE = (dSdu * dEdu).sum(dim=(-1, -2), keepdim=True)
            norm_E_sq = (dEdu * dEdu).sum(dim=(-1, -2), keepdim=True) + 1e-10
            dSdu_perp = dSdu - (inner_SE / norm_E_sq) * dEdu
            inner_S_Sperp = (dSdu * dSdu_perp).sum(dim=(-1, -2), keepdim=True) + 1e-10
            alpha = violation / inner_S_Sperp
            dudt = dudt + alpha * dSdu_perp
        return dudt

    def entropy_production(self, u):
        """Normalized entropy production r_S = <dS/du, du/dt> / (||dS/du|| ||du/dt||),
        fully differentiable. Used as a minimum-entropy-production (MEP) penalty:
        among GENERIC representations consistent with the data, prefer the one that
        produces the least entropy. Scale-invariant in S (can't be gamed by
        rescaling S), so reducing it requires genuinely making du/dt more
        orthogonal to dS/du -- i.e. genuinely less dissipative."""
        u_leaf = u.detach().requires_grad_(True)
        E = self.E_net(u_leaf); S = self.S_net(u_leaf)
        dEdu = torch.autograd.grad(E.sum(), u_leaf, create_graph=True)[0]
        dSdu = torch.autograd.grad(S.sum(), u_leaf, create_graph=True)[0]
        H, W = u.shape[-2], u.shape[-1]
        if self.l2_vargrad:
            scale = (H * W) / (2.0 * math.pi) ** 2
            dEdu = dEdu * scale
            dSdu = dSdu * scale
        if self.degeneracy_construction:
            rev, diss = self._generic_rhs(dEdu, dSdu, H, W)
            dudt = rev + diss
        else:
            dEh = torch.fft.rfft2(dEdu, dim=(-2, -1))
            dSh = torch.fft.rfft2(dSdu, dim=(-2, -1))
            rev_hat, diss_hat = self._apply_operators(dEh, dSh, H, W)
            rev = torch.fft.irfft2(rev_hat, s=(H, W))
            diss = torch.fft.irfft2(diss_hat, s=(H, W))
            dudt = rev + diss
            dudt = self._project_energy_conservation(dudt, dEdu)
            dudt = self._ensure_entropy_production(dudt, dSdu, dEdu)
            if self.use_residual:
                gate = torch.sigmoid(self.residual_gate)
                res = self._project_energy_conservation(gate * self.residual(u_leaf), dEdu)
                dudt = dudt + res
        num = (dSdu * dudt).flatten(1).sum(dim=1)
        den = dSdu.flatten(1).norm(dim=1) * dudt.flatten(1).norm(dim=1) + 1e-8
        return (num / den).clamp(min=0).mean()

    def predict_with_info(self, u):
        u_leaf = u.detach().requires_grad_(True)
        E = self.E_net(u_leaf)
        S = self.S_net(u_leaf)
        _ = torch.autograd.grad(E.sum(), u_leaf, create_graph=True)[0]
        _ = torch.autograd.grad(S.sum(), u_leaf, create_graph=True)[0]
        u_next = self.forward(u)
        u_next_leaf = u_next.detach().requires_grad_(True)
        E_next = self.E_net(u_next_leaf)
        S_next = self.S_net(u_next_leaf)
        info = {
            'E': E.detach(), 'S': S.detach(),
            'dE': (E_next - E).detach(), 'dS': (S_next - S).detach(),
        }
        return u_next, info

    def degeneracy_loss(self, u):
        """Soft regularizer: L·δS/δu ≈ 0 and M·δE/δu ≈ 0.
        Under degeneracy_construction these hold exactly, so the penalty is 0
        (kept only for the legacy projection path)."""
        if self.degeneracy_construction:
            return torch.zeros((), device=u.device)
        u_leaf = u.detach().requires_grad_(True)
        E = self.E_net(u_leaf)
        S = self.S_net(u_leaf)
        dEdu = torch.autograd.grad(E.sum(), u_leaf, create_graph=True)[0]
        dSdu = torch.autograd.grad(S.sum(), u_leaf, create_graph=True)[0]
        dEdu_hat = torch.fft.rfft2(dEdu, dim=(-2, -1))
        dSdu_hat = torch.fft.rfft2(dSdu, dim=(-2, -1))
        m1, m2 = self.m1, self.m2

        # L·δS/δu
        L_dS_pos = 1j * self.a_pos * dSdu_hat[:, :, :m1, :m2]
        L_dS_neg = 1j * self.a_neg * dSdu_hat[:, :, -m1:, :m2]
        loss_L = (L_dS_pos.abs()**2).mean() + (L_dS_neg.abs()**2).mean()

        # M·δE/δu
        M_pos = self.b_pos_r**2 + self.b_pos_i**2
        M_neg = self.b_neg_r**2 + self.b_neg_i**2
        M_dE_pos = M_pos * dEdu_hat[:, :, :m1, :m2]
        M_dE_neg = M_neg * dEdu_hat[:, :, -m1:, :m2]
        loss_M = (M_dE_pos.abs()**2).mean() + (M_dE_neg.abs()**2).mean()

        return loss_L + loss_M


# ============================================================================
# Training
# ============================================================================


# ============================================================================
# Evaluation
# ============================================================================



# ==================== PART C: LONG-HORIZON EXPERIMENT ====================
import os, math
import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ----------------------------- configuration --------------------------------
LH_RES    = int(os.environ.get("LH_RES",    128))   # train+eval resolution (128 = paper grid)
LH_NLONG  = int(os.environ.get("LH_NLONG",  200))   # rollout horizon (training used 15)
LH_NTRAIN = int(os.environ.get("LH_NTRAIN",  96))   # training trajectories
LH_NT     = int(os.environ.get("LH_NT",      15))   # training trajectory length
LH_EPOCHS = int(os.environ.get("LH_EPOCHS",  50))
LH_BATCH  = int(os.environ.get("LH_BATCH",    8))
LH_NTEST  = int(os.environ.get("LH_NTEST",    8))   # held-out test trajectories
LH_K      = int(os.environ.get("LH_K",        2))   # rollout horizon used in training
LH_LR     = 1e-3
LH_LAMBDA = 1.0                                      # EP-FNO energy-penalty weight
LH_PDES   = ["heat", "advection", "burgers"]
LH_KINDS  = ["FNO", "EP-FNO", "GENERIC"]
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
LH_FIGDIR = os.environ.get("GENERIC_FNO_FIGDIR",
                           os.path.join(os.environ.get("GENERIC_FNO_BASE", "."), "figures"))
LH_C  = {"FNO": "#7f7f7f", "EP-FNO": "#1f77b4", "GENERIC": "#d62728", "truth": "#000000"}
LH_LBL = {"heat": "Heat (dissipative)", "advection": "Advection (reversible)",
          "burgers": "Burgers (mixed)"}

# ------------------------------- helpers ------------------------------------
def _lh_grab():
    """Pull the model classes + generators from the notebook namespace (no files)."""
    G = globals()
    need = ["GENERIC_FNO2d", "VanillaFNO2d", "EP_FNO2d",
            "generate_heat_data_2d", "generate_advection_data_2d", "generate_burgers_data_2d"]
    miss = [n for n in need if G.get(n) is None]
    if miss:
        raise NameError("Missing " + ", ".join(miss) + " in the notebook namespace -- run "
                        "Part A (the model classes + generators) in an earlier cell first.")
    GEN = {"heat": G["generate_heat_data_2d"],
           "advection": G["generate_advection_data_2d"],
           "burgers": G["generate_burgers_data_2d"]}
    return G["GENERIC_FNO2d"], G["VanillaFNO2d"], G["EP_FNO2d"], GEN

def _is_generic(m):
    return hasattr(m, "E_net")

def _rel_l2(pred, tgt):
    p = pred.reshape(pred.shape[0], -1); t = tgt.reshape(tgt.shape[0], -1)
    return (torch.linalg.vector_norm(p - t, dim=1) /
            (torch.linalg.vector_norm(t, dim=1) + 1e-12)).mean()

def _make(kind, Model, Vanilla, EP):
    if kind == "GENERIC": return Model(nx=LH_RES, degeneracy_construction=True).to(DEVICE)
    if kind == "EP-FNO":  return EP().to(DEVICE)
    return Vanilla().to(DEVICE)

def _train(model, pde, kind, GEN):
    di, do, _ = GEN[pde](n_samples=LH_NTRAIN, nx=LH_RES, nt=LH_NT, device=DEVICE)
    di = di.to(DEVICE).float(); do = do.to(DEVICE).float()       # [B, T, H, W]
    B, T = di.shape[0], di.shape[1]
    opt = torch.optim.Adam(model.parameters(), lr=LH_LR)
    model.train()
    for ep in range(LH_EPOCHS):
        perm = torch.randperm(B, device=DEVICE)
        for i in range(0, B, LH_BATCH):
            idx = perm[i:i + LH_BATCH]
            di_b, do_b = di[idx], do[idx]
            t0 = int(torch.randint(0, T - LH_K + 1, (1,)).item())
            u = di_b[:, t0].unsqueeze(1)                          # [b, 1, H, W]
            loss = 0.0
            for k in range(LH_K):
                Qb = 0.5 * (u ** 2).mean(dim=(-1, -2))
                u = model(u)
                tgt = do_b[:, t0 + k].unsqueeze(1)
                loss = loss + _rel_l2(u, tgt)
                if kind == "EP-FNO":
                    dE = (0.5 * (u ** 2).mean(dim=(-1, -2)) - Qb).reshape(-1)
                    pen = (torch.relu(dE) ** 2).mean() if pde in ("heat", "burgers") else (dE ** 2).mean()
                    loss = loss + LH_LAMBDA * pen
            (loss / LH_K).backward()
            opt.step(); opt.zero_grad()
        if (ep + 1) % max(1, LH_EPOCHS // 5) == 0:
            print(f"      [{pde}/{kind}] epoch {ep+1}/{LH_EPOCHS}  loss={float(loss.detach())/LH_K:.4f}")
    model.eval()
    return model

@torch.no_grad()
def _rollout_fno(model, u0, n):
    u = u0.clone(); out = [u.clone()]
    for _ in range(n):
        u = model(u); out.append(u.clone())
    return torch.stack(out, dim=1)                                # [B, n+1, 1, H, W]

def _rollout_generic(model, u0, n):
    # GENERIC.forward differentiates E,S internally -> do NOT use no_grad
    u = u0.clone(); out = [u.detach().clone()]
    for _ in range(n):
        u = model(u).detach(); out.append(u.clone())
    return torch.stack(out, dim=1)

def _rollout(model, u0, n):
    return _rollout_generic(model, u0, n) if _is_generic(model) else _rollout_fno(model, u0, n)

def _metrics(pred, ref):
    """pred,ref: [B, S, H, W]. Returns (err_t, Qm_t, Qr_t) length-S arrays."""
    B, S = pred.shape[0], pred.shape[1]
    pf = pred.reshape(B, S, -1); rf = ref.reshape(B, S, -1)
    ref0 = torch.linalg.vector_norm(rf[:, 0], dim=1) + 1e-12             # [B]
    err = (torch.linalg.vector_norm(pf - rf, dim=2) / ref0[:, None]).mean(0)
    Qm = (0.5 * (pf ** 2).mean(dim=2)).mean(0)
    Qr = (0.5 * (rf ** 2).mean(dim=2)).mean(0)
    return err.cpu().numpy(), Qm.cpu().numpy(), Qr.cpu().numpy()

def _save(fig, name):
    os.makedirs(LH_FIGDIR, exist_ok=True)
    done = []
    for ext in ("pdf", "png"):
        p = os.path.join(LH_FIGDIR, f"{name}.{ext}")
        try:
            fig.savefig(p, bbox_inches="tight"); done.append(ext)
        except Exception as e:
            print(f"  [WARN] could not save {p}: {e}")
    plt.close(fig)
    print(f"  saved {name}: {'+'.join(done) if done else 'NOTHING'} -> {LH_FIGDIR}")

# ------------------------------- driver -------------------------------------
def run_long_horizon(pretrained=None, seed=0):
    """Train (or reuse) FNO / EP-FNO / GENERIC on each PDE and roll out LH_NLONG
    steps. pretrained: optional {(pde, kind): model}. Returns the results dict."""
    Model, Vanilla, EP, GEN = _lh_grab()
    torch.manual_seed(seed); np.random.seed(seed)
    pretrained = pretrained or {}
    print(f"Long-horizon experiment  res={LH_RES}  horizon={LH_NLONG} steps "
          f"(training horizon was {LH_NT})  device={DEVICE}")
    results = {}
    for pde in LH_PDES:
        print(f"  PDE: {pde}")
        # held-out long ground-truth trajectory (fresh ICs -> not in training set)
        di, do, _ = GEN[pde](n_samples=LH_NTEST, nx=LH_RES, nt=LH_NLONG, device=DEVICE)
        di = di.to(DEVICE).float(); do = do.to(DEVICE).float()
        ref = torch.cat([di[:, :1], do], dim=1)                  # [B, NLONG+1, H, W]
        u0 = di[:, 0].unsqueeze(1)                               # [B, 1, H, W]
        per_pde = {}
        for kind in LH_KINDS:
            m = pretrained.get((pde, kind))
            if m is not None:
                m = m.to(DEVICE).eval(); print(f"    {kind}: using provided model")
            else:
                print(f"    {kind}: training ...")
                m = _train(_make(kind, Model, Vanilla, EP), pde, kind, GEN)
            pred = _rollout(m, u0, LH_NLONG).squeeze(2)          # [B, NLONG+1, H, W]
            err, Qm, Qr = _metrics(pred, ref)
            per_pde[kind] = {"err": err, "Q": Qm}
            per_pde["truth_Q"] = Qr
            print(f"      {kind}: err@{LH_NT}step={err[LH_NT]:.3f}  err@{LH_NLONG}step={err[-1]:.3f}")
        results[pde] = per_pde
    _plot(results)
    return results

def _plot(results):
    n = len(LH_PDES)
    fig, axes = plt.subplots(n, 2, figsize=(11, 3.4 * n), squeeze=False)
    for r, pde in enumerate(LH_PDES):
        R = results[pde]; t = np.arange(len(R["truth_Q"]))
        Qr0 = R["truth_Q"][0] + 1e-12
        ax0, ax1 = axes[r, 0], axes[r, 1]
        for kind in LH_KINDS:
            ax0.semilogy(t, R[kind]["err"] + 1e-12, color=LH_C[kind], label=kind, lw=1.8)
            ax1.plot(t, R[kind]["Q"] / Qr0, color=LH_C[kind], label=kind, lw=1.8)
        ax1.plot(t, R["truth_Q"] / Qr0, "--", color=LH_C["truth"], label="ground truth", lw=1.5)
        for ax in (ax0, ax1):
            ax.axvline(LH_NT, color="black", ls=":", lw=1, alpha=0.6)
            ax.grid(ls=":", alpha=0.5); ax.set_xlabel("rollout step")
        ax0.set_ylabel(f"{LH_LBL[pde]}\nnormalized error")
        ax1.set_ylabel(r"physical energy $Q/Q_0$")
        if r == 0:
            ax0.set_title("Rollout error (log) — past the training horizon")
            ax1.set_title(r"Physical energy $Q=\frac{1}{2}\|u\|^2$ vs. truth")
            ax0.legend(fontsize=8, loc="upper left"); ax1.legend(fontsize=8, loc="best")
        ax0.annotate("train\nhorizon", xy=(LH_NT, ax0.get_ylim()[0]), fontsize=6,
                     color="black", alpha=0.6, ha="center", va="bottom")
    fig.suptitle("Long-horizon stability: GENERIC stays bounded and tracks the true energy "
                 f"far past the {LH_NT}-step training horizon", y=1.005, fontsize=11)
    fig.tight_layout()
    _save(fig, "fig_long_horizon")

if __name__ == "__main__":
    run_long_horizon()

Long-horizon experiment  res=128  horizon=200 steps (training horizon was 15)  device=cuda
  PDE: heat
    FNO: training ...
      [heat/FNO] epoch 10/50  loss=0.0229
      [heat/FNO] epoch 20/50  loss=0.0117
      [heat/FNO] epoch 30/50  loss=0.0143
      [heat/FNO] epoch 40/50  loss=0.0078
      [heat/FNO] epoch 50/50  loss=0.0091
      FNO: err@15step=0.158  err@200step=4.914
    EP-FNO: training ...
      [heat/EP-FNO] epoch 10/50  loss=0.0160
      [heat/EP-FNO] epoch 20/50  loss=0.0093
      [heat/EP-FNO] epoch 30/50  loss=0.0102
      [heat/EP-FNO] epoch 40/50  loss=0.0086
      [heat/EP-FNO] epoch 50/50  loss=0.0037
      EP-FNO: err@15step=0.201  err@200step=0.927
    GENERIC: training ...
      [heat/GENERIC] epoch 10/50  loss=0.0224
      [heat/GENERIC] epoch 20/50  loss=0.0237
      [heat/GENERIC] epoch 30/50  loss=0.0177
      [heat/GENERIC] epoch 40/50  loss=0.0163
      [heat/GENERIC] epoch 50/50  loss=0.0167
      GENERIC: err@15step=0.243  err@200step=0.750
  PDE: adve